# OpenStudio Parametric Study Workflow

This notebook runs energy simulations for various building envelope improvement scenarios.


In [4]:
# Standard library
import json
import os
import subprocess
import time
from itertools import product
import sys
from pathlib import Path
import configparser
import shutil
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
import csv

# Add OpenStudio 3.11.0 Python bindings to path BEFORE importing
def detect_openstudio_python_path():
    env_path = os.environ.get("OPENSTUDIO_PYTHON_PATH")
    candidates = [
        env_path,
        "C:/Program Files/openstudio-3.11.0/Python",
        "/Applications/OpenStudio-3.11.0/Python",
    ]
    for candidate in candidates:
        if candidate and os.path.exists(candidate):
            return candidate
    return None

OPENSTUDIO_PYTHON_PATH = detect_openstudio_python_path()
if OPENSTUDIO_PYTHON_PATH and OPENSTUDIO_PYTHON_PATH not in sys.path:
    sys.path.insert(0, OPENSTUDIO_PYTHON_PATH)
elif not OPENSTUDIO_PYTHON_PATH:
    print("Warning: OpenStudio Python path not found; trying system package.")
import openstudio

# Read EC3 API Token from config.ini
def get_ec3_api_token():
    """Read EC3 API token from config.ini file."""
    script_dir = Path.cwd()
    repo_root = script_dir.parent.parent
    config_path = repo_root / "config.ini"
    
    if not config_path.exists():
        print(f"Warning: config.ini not found at {config_path}")
        return None
    
    config = configparser.ConfigParser()
    config.read(config_path)
    
    try:
        return config["EC3_API_TOKEN"]["API_TOKEN"]
    except KeyError:
        print("Warning: EC3_API_TOKEN not found in config.ini")
        return None

EC3_API_TOKEN = get_ec3_api_token()


# =========================
# HELPER FUNCTIONS
# =========================

def get_city_weather_files(city_name, base_weather_path):
    """Get the EPW and DDY file paths for a given city name."""
    city_folder_path = os.path.join(base_weather_path, city_name)
    if not os.path.exists(city_folder_path):
        print(f"Error: Folder '{city_folder_path}' does not exist")
        return None

    epw_file = None
    ddy_file = None
    for filename in os.listdir(city_folder_path):
        if filename.lower().endswith(".epw"):
            epw_file = os.path.join(city_folder_path, filename)
        elif filename.lower().endswith(".ddy"):
            ddy_file = os.path.join(city_folder_path, filename)

    return {"epw": epw_file, "ddy": ddy_file}


def generate_scenario_name(scenario_dict):
    """Generate a unique scenario name from scenario parameters."""
    parts = []
    
    if scenario_dict.get("is_baseline", False):
        parts.append("baseline")
    else:
        measure_count = sum([
            1 if scenario_dict.get("wall_r_value") else 0,
            1 if scenario_dict.get("roof_r_value") else 0,
            1 if scenario_dict.get("window_u_factor") else 0,
            1 if scenario_dict.get("door_option") else 0,
        ])
        
        if measure_count > 1:
            parts.append("all")
        
        if scenario_dict.get("wall_r_value"):
            parts.append(f"wall_r{scenario_dict['wall_r_value']}")
        if scenario_dict.get("roof_r_value"):
            parts.append(f"roof_r{scenario_dict['roof_r_value']}")
        if scenario_dict.get("window_u_factor"):
            parts.append(f"window_u{scenario_dict['window_u_factor']}")
        if scenario_dict.get("door_option"):
            door_abbrev = scenario_dict['door_option'].replace(' ', '_').replace('door', 'd')
            parts.append(f"door_{door_abbrev}")
    
    parts.append(scenario_dict["building_type"])
    parts.append(scenario_dict["city"])
    
    return "_".join(parts)


def run_osw(osw_dict, osw_filename, run_dir, openstudio_path, label):
    """Write an OSW and run it with the OpenStudio CLI. Returns True on success."""
    osw_path = os.path.join(run_dir, osw_filename)
    os.makedirs(run_dir, exist_ok=True)
    with open(osw_path, "w") as f:
        json.dump(osw_dict, f, indent=2)

    try:
        result = subprocess.run(
            [openstudio_path, "run", "-w", osw_filename],
            check=False,
            capture_output=True,
            text=True,
            cwd=run_dir,
            timeout=600,
        )
    except subprocess.TimeoutExpired:
        print(f"❌ {label}: timed out after 600s")
        return False
    except Exception as e:
        print(f"❌ {label}: subprocess error: {e}")
        return False

    out_osw_path = os.path.join(run_dir, "out.osw")
    if not os.path.exists(out_osw_path):
        print(f"❌ {label}: out.osw not found (OpenStudio may have crashed)")
        if result.stderr:
            print(f"   STDERR: {result.stderr[:500]}")
        return False

    with open(out_osw_path, "r") as f:
        out_osw = json.load(f)

    if out_osw.get("completed_status") != "Success":
        print(f"❌ {label}: OSW failed")
        run_log_path = os.path.join(run_dir, "run", "run.log")
        if os.path.exists(run_log_path):
            with open(run_log_path, "r") as log_f:
                for line in log_f:
                    if "ERROR" in line:
                        print(f"   LOG: {line.rstrip()}")
        return False

    return True


def apply_python_measure(model, measure_folder, measure_class_name, arguments_dict):
    """
    Apply a Python OpenStudio ModelMeasure directly to a model in-process.
    Returns True if successful, False otherwise.
    """
    measure_folder_str = str(measure_folder)
    try:
        if measure_folder_str not in sys.path:
            sys.path.insert(0, measure_folder_str)

        import measure as measure_module
        import importlib
        importlib.reload(measure_module)

        measure_class = getattr(measure_module, measure_class_name)

        osw = openstudio.WorkflowJSON()
        runner = openstudio.measure.OSRunner(osw)
        measure = measure_class()

        args = measure.arguments(model)
        arg_map = openstudio.measure.convertOSArgumentVectorToMap(args)

        for arg_name, arg_value in arguments_dict.items():
            if arg_name in arg_map:
                arg = arg_map[arg_name]
                arg.setValue(arg_value)
                arg_map[arg_name] = arg

        measure.run(model, runner, arg_map)

        result_value = runner.result().value().valueName()
        if result_value != "Success":
            print(f"  Measure result: {result_value}")
            for error in runner.result().errors():
                print(f"    ERROR: {error.logMessage()}")
            return False

        return True

    except Exception as e:
        print(f"  ERROR applying measure: {str(e)}")
        import traceback
        traceback.print_exc()
        return False
    finally:
        if measure_folder_str in sys.path:
            sys.path.remove(measure_folder_str)


def apply_reporting_measure(model_path, sql_file_path, measure_dir_path, label):
    """
    Apply the OperatingCostCarbonReportingMeasure (Python ReportingMeasure)
    in-process after simulation completes. The OSW runner can't execute Python
    measures, so we call it directly using the OpenStudio Python bindings.

    Returns True if successful, False otherwise.
    """
    measure_folder = os.path.join(measure_dir_path, "OperatingCostCarbonReportingMeasure")
    measure_folder_str = str(measure_folder)
    try:
        # Load model
        translator = openstudio.osversion.VersionTranslator()
        loaded = translator.loadModel(openstudio.toPath(str(model_path)))
        if not loaded.is_initialized():
            print(f"  ⚠️  {label}: could not load model for reporting measure")
            return False
        model = loaded.get()

        # Attach SQL file to model
        sql_file = openstudio.SqlFile(openstudio.toPath(str(sql_file_path)))
        model.setSqlFile(sql_file)

        # Set up runner with SQL file and model
        osw = openstudio.WorkflowJSON()
        runner = openstudio.measure.OSRunner(osw)
        runner.setLastEnergyPlusSqlFilePath(openstudio.toPath(str(sql_file_path)))
        runner.setLastOpenStudioModel(model)

        # Import and instantiate the measure
        if measure_folder_str not in sys.path:
            sys.path.insert(0, measure_folder_str)
        import measure as measure_module
        import importlib
        importlib.reload(measure_module)
        measure = measure_module.OperatingCostCarbonReport()

        # Arguments (none required for this measure — reads from CSV resources)
        args = measure.arguments(model)
        arg_map = openstudio.measure.convertOSArgumentVectorToMap(args)

        # Run
        measure.run(runner, arg_map)

        result_value = runner.result().value().valueName()
        if result_value != "Success":
            print(f"  ⚠️  {label}: reporting measure result: {result_value}")
            for error in runner.result().errors():
                print(f"    ERROR: {error.logMessage()}")
            return False

        # Save model with AdditionalProperties written by the reporting measure
        model.save(openstudio.toPath(str(model_path)), True)
        del model

        return True

    except Exception as e:
        print(f"  ⚠️  {label}: reporting measure error: {e}")
        import traceback
        traceback.print_exc()
        return False
    finally:
        if measure_folder_str in sys.path:
            sys.path.remove(measure_folder_str)


# =========================
# CORE: SINGLE SCENARIO CREATION/RUN
# =========================

def create_simulation(
    city,
    base_run_dir,
    measure_dir_path,
    base_weather_path,
    scenario_dict,
    overwrite_existing=False,
    building_type="SmallOffice",
    template="90.1-2013",
    climate_zone="ASHRAE 169-2013-5A",
    openstudio_path="openstudio",
):
    # --- Weather ---
    wf = get_city_weather_files(city, base_weather_path)
    if wf is None or wf["epw"] is None:
        print(f"❌ Weather files not found for {city}")
        return None

    epw_path = os.path.abspath(wf["epw"])
    scenario_name = generate_scenario_name(scenario_dict)
    scenario_run_dir = os.path.abspath(os.path.join(base_run_dir, scenario_name))
    os.makedirs(scenario_run_dir, exist_ok=True)

    # Skip if already done
    sql_output_path = os.path.join(scenario_run_dir, "run", "eplusout.sql")
    if os.path.exists(sql_output_path) and not overwrite_existing:
        print(f"⏭️  Skipping {scenario_name} - simulation already exists")
        return scenario_name

    measure_paths = [os.path.abspath(measure_dir_path)]
    file_paths = [os.path.abspath(base_weather_path), os.path.dirname(epw_path)]

    prototype_step = {
        "measure_dir_name": "create_DOE_prototype_building",
        "name": "Create DOE Prototype Building",
        "arguments": {
            "building_type": building_type,
            "template": template,
            "climate_zone": climate_zone,
            "epw_file": "Not Applicable",
        },
    }

    # ====================================================================
    # BASELINE: OSW with prototype only → E+ simulation runs automatically
    # Then apply Python reporting measure in-process.
    # ====================================================================
    if scenario_dict.get("is_baseline", False):
        osw = {
            "weather_file": epw_path,
            "file_paths": file_paths,
            "measure_paths": measure_paths,
            "steps": [prototype_step],
            "name": scenario_name,
        }
        success = run_osw(osw, "run.osw", scenario_run_dir, openstudio_path, scenario_name)
        if not success:
            return None

        # Apply Python reporting measure after simulation
        model_path = os.path.join(scenario_run_dir, "run", "in.osm")
        sql_path = os.path.join(scenario_run_dir, "run", "eplusout.sql")
        if os.path.exists(sql_path):
            print(f"  Applying reporting measure...")
            apply_reporting_measure(model_path, sql_path, measure_dir_path, scenario_name)

    # ====================================================================
    # NON-BASELINE:
    #   Phase 1 — create prototype in a _proto/ subfolder
    #   Phase 2 — apply Python model measures in-process
    #   Phase 3 — OSW with seed_file (no measure steps) → E+ simulation
    #   Phase 4 — apply Python reporting measure in-process
    # ====================================================================
    else:
        # --- Phase 1: create prototype in isolated subfolder ---
        proto_dir = os.path.join(scenario_run_dir, "_proto")
        proto_osw = {
            "weather_file": epw_path,
            "file_paths": file_paths,
            "measure_paths": measure_paths,
            "steps": [prototype_step],
            "name": f"{scenario_name}_proto",
        }
        success = run_osw(proto_osw, "proto.osw", proto_dir, openstudio_path, f"{scenario_name} [prototype]")
        if not success:
            return None

        proto_model_path = os.path.join(proto_dir, "run", "in.osm")
        if not os.path.exists(proto_model_path):
            print(f"❌ {scenario_name}: prototype model not found at {proto_model_path}")
            return None

        final_model_path = os.path.join(scenario_run_dir, "model_to_run.osm")
        shutil.copy2(proto_model_path, final_model_path)

        # --- Phase 2: apply Python model measures ---
        translator = openstudio.osversion.VersionTranslator()
        loaded_model = translator.loadModel(openstudio.toPath(final_model_path))

        if not loaded_model.is_initialized():
            print(f"❌ {scenario_name}: failed to load prototype model")
            return None

        model = loaded_model.get()

        if scenario_dict.get("wall_r_value"):
            wall_args = {
                "r_value": float(scenario_dict["wall_r_value"]),
                "analysis_period": 30,
                "gwp_statistic": "median",
                "api_key": EC3_API_TOKEN or "",
                "insulation_material_type": "Blown Fiberglass",
                "insulation_material_lifetime": 30,
                "insulation_thermal_conductivity": 0.0,
                "insulation_material_density": 0.0,
            }
            print(f"  Applying wall insulation (R={scenario_dict['wall_r_value']})...")
            if not apply_python_measure(model, Path(measure_dir_path) / "IncreaseInsulationRValueForExteriorWalls", "IncreaseInsulationRValueForExteriorWalls", wall_args):
                print(f"❌ {scenario_name}: wall measure failed")
                del model
                return None

        if scenario_dict.get("roof_r_value"):
            roof_args = {
                "r_value": float(scenario_dict["roof_r_value"]),
                "analysis_period": 30,
                "gwp_statistic": "median",
                "api_key": EC3_API_TOKEN or "",
                "insulation_material_type": "Blown Fiberglass",
                "insulation_material_lifetime": 30,
                "insulation_thermal_conductivity": 0.0,
                "insulation_material_density": 0.0,
            }
            print(f"  Applying roof insulation (R={scenario_dict['roof_r_value']})...")
            if not apply_python_measure(model, Path(measure_dir_path) / "IncreaseInsulationRValueForRoofs", "IncreaseInsulationRValueForRoofs", roof_args):
                print(f"❌ {scenario_name}: roof measure failed")
                del model
                return None

        if scenario_dict.get("window_u_factor"):
            if EC3_API_TOKEN is None:
                print(f"⚠️  EC3 API token not found, skipping window enhancement")
            else:
                u_factor = float(scenario_dict["window_u_factor"])
                num_panes = 2 if u_factor >= 0.30 else 3
                window_args = {
                    "glass_option": "provide user_num_panes",
                    "user_num_panes": num_panes,
                    "space_infiltration_reduction_percent": 50.0,
                    "glass_pane_thickness": 0.003,
                    "gap_thickness": 0.013,
                    "glass_solar_transmittance": 0.7,
                    "glass_visible_transmittance": 0.8,
                    "glass_front_emissivity": 0.84,
                    "glass_back_emissivity": 0.84,
                    "glass_front_solar_reflectance": 0.15,
                    "glass_back_solar_reflectance": 0.15,
                    "glass_front_visible_reflectance": 0.1,
                    "glass_back_visible_reflectance": 0.1,
                    "analysis_period": 30,
                    "glass_lifetime": 15,
                    "wf_lifetime": 15,
                    "caulking_lifetime": 10,
                    "film_lifetime": 10,
                    "weatherstrip_lifetime": 10,
                    "wf_option": "none",
                    "caulking_option": "none",
                    "caulking_thickness": 0.003,
                    "film_option": "none",
                    "film_visible_transmittance": 0.0,
                    "film_solar_transmittance": 0.0,
                    "film_thermal_emissivity": 0.0,
                    "film_thermal_resistance": 0.0,
                    "weatherstrip_option": "none",
                    "length_per_unit": 1.0,
                    "secondary_glazing_option": "none",
                    "api_key": EC3_API_TOKEN,
                    "gwp_statistic": "median",
                }
                print(f"  Applying window enhancement (U={scenario_dict['window_u_factor']}, {num_panes} panes)...")
                if not apply_python_measure(model, Path(measure_dir_path) / "window_enhancement", "WindowEnhancement", window_args):
                    print(f"❌ {scenario_name}: window measure failed")
                    del model
                    return None

        if scenario_dict.get("door_option"):
            if EC3_API_TOKEN is None:
                print(f"⚠️  EC3 API token not found, skipping door enhancement")
            else:
                door_args = {
                    "space_infiltration_reduction_percent": 30.0,
                    "alter_coef": False,
                    "door_area_per_unit": 1.95,
                    "analysis_period": 30,
                    "door_bottom_seal_option": "automatic door bottom",
                    "door_top_side_seal_option": "jamb weatherstrip",
                    "door_option": scenario_dict["door_option"],
                    "strip_lifetime": 15,
                    "door_lifetime": 30,
                    "gwp_statistic": "median",
                    "api_key": EC3_API_TOKEN,
                    "length_per_unit_bottom_side": 0.9144,
                    "length_per_unit_other_sides": 5.1816,
                    "door_thermal_conductivity": 0.0,
                    "door_density": 0.0,
                    "door_thickness": 0.0,
                }
                print(f"  Applying door enhancement (door={scenario_dict['door_option']})...")
                if not apply_python_measure(model, Path(measure_dir_path) / "door_enhancement", "DoorEnhancement", door_args):
                    print(f"❌ {scenario_name}: door measure failed")
                    del model
                    return None

        model.save(openstudio.toPath(final_model_path), True)
        del model

        # --- Phase 3: OSW with seed to run EnergyPlus (no measure steps) ---
        sim_osw = {
            "weather_file": epw_path,
            "seed_file": final_model_path,
            "file_paths": file_paths,
            "measure_paths": measure_paths,
            "steps": [],
            "name": scenario_name,
        }
        success = run_osw(sim_osw, "run.osw", scenario_run_dir, openstudio_path, scenario_name)
        if not success:
            return None

        # --- Phase 4: Apply Python reporting measure after simulation ---
        model_path = os.path.join(scenario_run_dir, "run", "in.osm")
        sql_path = os.path.join(scenario_run_dir, "run", "eplusout.sql")
        if os.path.exists(sql_path):
            print(f"  Applying reporting measure...")
            apply_reporting_measure(model_path, sql_path, measure_dir_path, scenario_name)

    print(f"✅ Completed: {scenario_name}")
    return scenario_name

# =========================
# SCENARIO GENERATION
# =========================

def generate_scenarios(
    cities,
    building_types,
    run_wall_insulation=False,
    run_roof_insulation=False,
    run_window_enhancement=False,
    run_door_enhancement=False,
    run_all_measures=False,
    wall_r_values=None,
    roof_r_values=None,
    window_u_factors=None,
    door_options=None,
    custom_combos=None,
):
    """
    Generate scenarios:
    - Always includes baseline
    - Individual measures (wall only, roof only, window only, door only)
    - All measures combined (Cartesian product)
    - Custom explicit combinations via custom_combos
    """
    scenarios = []

    # 1) BASELINE
    for city, building_type in product(cities, building_types):
        scenarios.append({
            "is_baseline": True,
            "city": city,
            "building_type": building_type,
            "wall_r_value": None,
            "roof_r_value": None,
            "window_u_factor": None,
            "door_option": None,
        })

    # 2) INDIVIDUAL MEASURES
    if run_wall_insulation and wall_r_values:
        for city, building_type, wall_r in product(cities, building_types, wall_r_values):
            scenarios.append({
                "is_baseline": False,
                "city": city,
                "building_type": building_type,
                "wall_r_value": wall_r,
                "roof_r_value": None,
                "window_u_factor": None,
                "door_option": None,
            })

    if run_roof_insulation and roof_r_values:
        for city, building_type, roof_r in product(cities, building_types, roof_r_values):
            scenarios.append({
                "is_baseline": False,
                "city": city,
                "building_type": building_type,
                "wall_r_value": None,
                "roof_r_value": roof_r,
                "window_u_factor": None,
                "door_option": None,
            })

    if run_window_enhancement and window_u_factors:
        for city, building_type, window_u in product(cities, building_types, window_u_factors):
            scenarios.append({
                "is_baseline": False,
                "city": city,
                "building_type": building_type,
                "wall_r_value": None,
                "roof_r_value": None,
                "window_u_factor": window_u,
                "door_option": None,
            })

    if run_door_enhancement and door_options:
        for city, building_type, door_opt in product(cities, building_types, door_options):
            scenarios.append({
                "is_baseline": False,
                "city": city,
                "building_type": building_type,
                "wall_r_value": None,
                "roof_r_value": None,
                "window_u_factor": None,
                "door_option": door_opt,
            })

    # 3) ALL MEASURES COMBINED (Cartesian product)
    if run_all_measures:
        walls = wall_r_values if wall_r_values else [None]
        roofs = roof_r_values if roof_r_values else [None]
        windows = window_u_factors if window_u_factors else [None]
        doors = door_options if door_options else [None]

        for city, building_type, wall_r, roof_r, window_u, door_opt in product(
            cities, building_types, walls, roofs, windows, doors
        ):
            if wall_r or roof_r or window_u or door_opt:
                scenarios.append({
                    "is_baseline": False,
                    "city": city,
                    "building_type": building_type,
                    "wall_r_value": wall_r,
                    "roof_r_value": roof_r,
                    "window_u_factor": window_u,
                    "door_option": door_opt,
                })

    # 4) CUSTOM EXPLICIT COMBINATIONS
    if custom_combos:
        for city, building_type, combo in product(cities, building_types, custom_combos):
            scenarios.append({
                "is_baseline": False,
                "city": city,
                "building_type": building_type,
                "wall_r_value": combo.get("wall_r_value"),
                "roof_r_value": combo.get("roof_r_value"),
                "window_u_factor": combo.get("window_u_factor"),
                "door_option": combo.get("door_option"),
            })

    return scenarios


# =========================
# POSTPROCESS: COLLECT RESULTS
# =========================

def _read_props_to_dict(props, prefix=""):
    """Helper: read all features from an AdditionalProperties object into a dict."""
    result = {}
    for name in props.featureNames():
        val = props.getFeatureAsString(name)
        if val.is_initialized():
            v = val.get()
            try:
                result[prefix + name] = float(v)
            except ValueError:
                result[prefix + name] = v
    return result


def extract_additional_properties_from_osm(osm_path):
    """
    Extract AdditionalProperties from an OSM file.
    Reads from: Building, Site, Facility, SimulationControl, SizingParameters.
    Returns a flat dict of all found properties.
    """
    try:
        translator = openstudio.osversion.VersionTranslator()
        translator.setAllowNewerVersions(True)
        loaded_model = translator.loadModel(openstudio.toPath(str(osm_path)))

        if not loaded_model.is_initialized():
            return {}

        model = loaded_model.get()
        prop_dict = {}

        # Building — measure inputs (measure_name, analysis_period_years, gwp_statistic, etc.)
        prop_dict.update(_read_props_to_dict(model.getBuilding().additionalProperties()))

        # Site — reno details + operating cost/emissions from reporting measure
        prop_dict.update(_read_props_to_dict(model.getSite().additionalProperties()))

        # Facility — GWP factors
        prop_dict.update(_read_props_to_dict(model.getFacility().additionalProperties()))

        # SimulationControl — embodied carbon results
        prop_dict.update(_read_props_to_dict(model.getSimulationControl().additionalProperties()))

        # SizingParameters — material properties (lifetimes, densities, etc.)
        prop_dict.update(_read_props_to_dict(model.getSizingParameters().additionalProperties()))

        del model
        return prop_dict

    except Exception as e:
        print(f"  ⚠️  Failed to extract properties from {osm_path}: {e}")
        return {}


def collect_results_to_csv(base_run_dir, csv_base_name="parametric_results"):
    """
    Walk all run directories, extract AdditionalProperties from OSM files,
    and collect all results into a comprehensive CSV.
    """
    import pandas as pd
    rows = []

    for scenario_folder in sorted(os.listdir(base_run_dir)):
        scenario_path = os.path.join(base_run_dir, scenario_folder)
        if not os.path.isdir(scenario_path):
            continue

        sql_path = os.path.join(scenario_path, "run", "eplusout.sql")
        if not os.path.exists(sql_path):
            continue

        scenario_parts = scenario_folder.split("_")
        is_baseline = scenario_parts[0] == "baseline"

        row = {
            "scenario_name": scenario_folder,
            "is_baseline": is_baseline,
        }

        osm_path = os.path.join(scenario_path, "run", "in.osm")
        if os.path.exists(osm_path):
            print(f"  Extracting properties from {scenario_folder}...")
            row.update(extract_additional_properties_from_osm(osm_path))

        rows.append(row)

    if not rows:
        print("\nℹ️ No simulation results found.")
        return None

    df_results = pd.DataFrame(rows)

    priority_cols = ["scenario_name", "is_baseline"]
    existing_priority = [c for c in priority_cols if c in df_results.columns]
    remaining = sorted([c for c in df_results.columns if c not in existing_priority])
    df_results = df_results[existing_priority + remaining]

    csv_path = os.path.join(base_run_dir, f"{csv_base_name}.csv")
    df_results.to_csv(csv_path, index=False)
    print(f"\n🧾 Results CSV: {csv_path}")
    print(f"   Total scenarios: {len(df_results)}")
    print(f"   Columns: {len(df_results.columns)}")

    return df_results


def get_prop_value(props, name):
    """Helper to safely extract value from AdditionalProperties by type."""
    if props.getFeatureAsDouble(name).is_initialized():
        return props.getFeatureAsDouble(name).get()
    if props.getFeatureAsString(name).is_initialized():
        return props.getFeatureAsString(name).get()
    if props.getFeatureAsInteger(name).is_initialized():
        return props.getFeatureAsInteger(name).get()
    return None


def extract_scenario_data(osm_path, scenario_name):
    """
    Loads an OSM and extracts target properties from AdditionalProperties.

    Key mapping (new measure schema):
      Embodied carbon  → SimulationControl.additionalProperties()
        wall_insulation_total_additional_embodied_carbon_kg
        roof_insulation_total_additional_embodied_carbon_kg
        window_enhancement_total_additional_embodied_carbon_kg
        door_enhancement_total_additional_embodied_carbon_kg
      Operating + reno detail data   → Site.additionalProperties()
        annual_electricity_cost_usd
        annual_gas_cost_usd
        annual_electricity_operating_emissions_kg_co2e
        annual_gas_operating_emissions_kg_co2e
        total_renovated_window_area_m2, etc.
    """
    results = {"scenario": scenario_name}

    vt = openstudio.osversion.VersionTranslator()
    model_ptr = vt.loadModel(openstudio.toPath(str(osm_path)))

    if not model_ptr.is_initialized():
        print(f"  ✗ Failed to load: {osm_path.name}")
        return None

    model = model_ptr.get()
    found_any = False

    # 1. Embodied Carbon — SimulationControl.additionalProperties()
    sim_props = model.getSimulationControl().additionalProperties()
    ec_keys = [
        "wall_insulation_total_additional_embodied_carbon_kg",
        "roof_insulation_total_additional_embodied_carbon_kg",
        "window_enhancement_total_additional_embodied_carbon_kg",
        "door_enhancement_total_additional_embodied_carbon_kg",
    ]
    for key in ec_keys:
        if key in sim_props.featureNames():
            val = get_prop_value(sim_props, key)
            if val is not None:
                results[key] = val
                found_any = True

    # 2. Operating Cost, Emissions, and Reno Details — Site.additionalProperties()
    site_props = model.getSite().additionalProperties()
    op_keys = [
        "annual_electricity_cost_usd",
        "annual_gas_cost_usd",
        "annual_electricity_operating_emissions_kg_co2e",
        "annual_gas_operating_emissions_kg_co2e",
    ]
    for key in op_keys:
        if key in site_props.featureNames():
            val = get_prop_value(site_props, key)
            if val is not None:
                results[key] = val
                found_any = True

    # Explicit reno detail keys written by envelope measures
    reno_keys = [
        "total_renovated_window_area_m2",
        "total_renovated_glazing_area_m2",
        "total_renovated_frame_area_m2",
        "total_renovated_perimeter_m",
        "total_renovated_caulking_volume_m3",
        "total_renovated_weatherstrip_length_m",
        "total_renovated_door_area_m2",
        "total_renovated_sealing_bottom_length_m",
        "total_renovated_sealing_side_length_m",
    ]
    for key in reno_keys:
        if key in site_props.featureNames():
            val = get_prop_value(site_props, key)
            if val is not None:
                results[key] = val
                found_any = True

    return results if found_any else None


def generate_parametric_recap(target_path):
    """
    Main function to run the extraction and generate parametric_results.csv
    """
    root_path = Path(target_path)

    if not root_path.exists():
        print(f"Error: Path '{target_path}' does not exist.")
        return

    all_data = []
    all_headers = set()

    print("=" * 80)
    print(f"GENERATING PARAMETRIC RECAP FROM: {root_path}")
    print("=" * 80)

    osm_files = [p for p in root_path.rglob("*.osm") if p.name in ["in.osm", "in_modified.osm"]]

    for osm_path in sorted(osm_files):
        parts = list(osm_path.parts)
        try:
            idx = parts.index('run')
            scenario = parts[idx - 1]
        except ValueError:
            scenario = osm_path.parent.name

        data = extract_scenario_data(osm_path, scenario)
        if data:
            all_data.append(data)
            all_headers.update(data.keys())

    if not all_data:
        print("\n✗ No matching data found.")
        return

    fixed_headers = [
        "scenario",
        "annual_electricity_cost_usd",
        "annual_gas_cost_usd",
        "annual_electricity_operating_emissions_kg_co2e",
        "annual_gas_operating_emissions_kg_co2e",
        "total_renovated_window_area_m2",
        "total_renovated_glazing_area_m2",
        "total_renovated_frame_area_m2",
        "total_renovated_perimeter_m",
        "total_renovated_caulking_volume_m3",
        "total_renovated_weatherstrip_length_m",
        "total_renovated_door_area_m2",
        "total_renovated_sealing_bottom_length_m",
        "total_renovated_sealing_side_length_m",
        "wall_insulation_total_additional_embodied_carbon_kg",
        "roof_insulation_total_additional_embodied_carbon_kg",
        "window_enhancement_total_additional_embodied_carbon_kg",
        "door_enhancement_total_additional_embodied_carbon_kg",
    ]
    extra_headers = sorted([h for h in all_headers if h not in fixed_headers and h != "scenario"])
    fieldnames = [h for h in fixed_headers if h in all_headers | {"scenario"}] + extra_headers

    csv_path = root_path / "parametric_results.csv"
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(all_data)

    print("\n" + "=" * 80)
    print(f"COMPLETE: {len(all_data)} scenarios successfully processed.")
    print(f"Report saved to: {csv_path}")
    print("=" * 80)


# =========================
# GLOBAL SETTINGS
# =========================

RUN_NAME = "run_test_001"

def detect_openstudio_cli_path():
    env_path = os.environ.get("OPENSTUDIO_PATH")
    candidates = [
        env_path,
        "C:/Program Files/openstudio-3.11.0/bin/openstudio.exe",
        "/Applications/OpenStudio-3.11.0/bin/openstudio",
        "openstudio",
    ]
    for candidate in candidates:
        if not candidate:
            continue
        if candidate == "openstudio" or os.path.exists(candidate):
            return candidate
    return "openstudio"

OPENSTUDIO_PATH = detect_openstudio_cli_path()
OVERWRITE_EXISTING = True

notebook_dir = Path.cwd()
base_weather_path = str(notebook_dir / "weather")
measure_dir_path = str(notebook_dir.parent / "measures")
base_run_dir = str(notebook_dir / "simulations" / RUN_NAME)

city_climate_zones = {
    "Amarillo":     "ASHRAE 169-2013-3B",
    # "Atlanta":      "ASHRAE 169-2013-3A",
    # "Baltimore":    "ASHRAE 169-2013-4A",
    # "Chicago":      "ASHRAE 169-2013-5A",
    # "Denver":       "ASHRAE 169-2013-5B",
    # "Duluth":       "ASHRAE 169-2013-7A",
    # "ElPaso":       "ASHRAE 169-2013-3B",
    # "Fairbanks":    "ASHRAE 169-2013-8A",
    # "Helena":       "ASHRAE 169-2013-6B",
    # "Houston":      "ASHRAE 169-2013-2A",
    # "Miami":        "ASHRAE 169-2013-1A",
    # "Minneapolis":  "ASHRAE 169-2013-6A",
    # "Phoenix":      "ASHRAE 169-2013-2B",
    # "PortAngeles":  "ASHRAE 169-2013-4C",
    # "Portland":     "ASHRAE 169-2013-4C",
    # "SanFrancisco": "ASHRAE 169-2013-3C",
}

# =========================
# PARAMETRIC STUDY CONFIGURATION
# =========================

RUN_WALL_INSULATION = False
RUN_ROOF_INSULATION = False
RUN_WINDOW_ENHANCEMENT = False
RUN_DOOR_ENHANCEMENT = False
RUN_ALL_MEASURES = False   # Handled explicitly in CUSTOM_COMBOS below

CITIES = list(city_climate_zones.keys())

BUILDING_TYPES = [
    "SmallOffice",
    # "MediumOffice",
    # "LargeOffice",
    # "SmallHotel",
    # "LargeHotel",
    # "Warehouse",
    # "RetailStandalone",
    # "RetailStripmall",
    # "PrimarySchool",
    # "SecondarySchool",
]

TEMPLATE = "90.1-2010"

WALL_R_VALUES = [30]
ROOF_R_VALUES = [40]
WINDOW_U_FACTORS = [0.20]
DOOR_OPTIONS = ["wooden door"]

# =========================
# CUSTOM COMBINATION SCENARIOS
# Scenario 1: Baseline  — auto-generated (no entry needed here)
# Scenario 2: Wall + Door
# Scenario 3: Roof + Window
# Scenario 4: Wall + Door + Roof + Window (all 4 measures)
# =========================
CUSTOM_COMBOS = [
    # Scenario 2: Wall + Door only
    {
        "wall_r_value":    30,
        "roof_r_value":    None,
        "window_u_factor": None,
        "door_option":     "wooden door",
    },
    # Scenario 3: Roof + Window only
    {
        "wall_r_value":    None,
        "roof_r_value":    40,
        "window_u_factor": 0.20,
        "door_option":     None,
    },
    # Scenario 4: All 4 measures — Wall + Door + Roof + Window
    {
        "wall_r_value":    30,
        "roof_r_value":    40,
        "window_u_factor": 0.20,
        "door_option":     "wooden door",
    },
]

# =========================
# MAIN - RUN PARAMETRIC STUDY
# =========================

if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("PARAMETRIC STUDY: BUILDING ENERGY EFFICIENCY MEASURES")
    print(f"Using OpenStudio: {OPENSTUDIO_PATH}")
    print(f"Run Name: {RUN_NAME}")
    print(f"Output Directory: {base_run_dir}")
    print("=" * 70)

    print("\n📋 Generating scenarios...")
    scenarios = generate_scenarios(
        cities=CITIES,
        building_types=BUILDING_TYPES,
        run_wall_insulation=RUN_WALL_INSULATION,
        run_roof_insulation=RUN_ROOF_INSULATION,
        run_window_enhancement=RUN_WINDOW_ENHANCEMENT,
        run_door_enhancement=RUN_DOOR_ENHANCEMENT,
        run_all_measures=RUN_ALL_MEASURES,
        wall_r_values=WALL_R_VALUES if RUN_WALL_INSULATION or RUN_ALL_MEASURES else None,
        roof_r_values=ROOF_R_VALUES if RUN_ROOF_INSULATION or RUN_ALL_MEASURES else None,
        window_u_factors=WINDOW_U_FACTORS if RUN_WINDOW_ENHANCEMENT or RUN_ALL_MEASURES else None,
        door_options=DOOR_OPTIONS if RUN_DOOR_ENHANCEMENT or RUN_ALL_MEASURES else None,
        custom_combos=CUSTOM_COMBOS,
    )

    total_sims = len(scenarios)
    baseline_count = sum(1 for s in scenarios if s["is_baseline"])
    individual_wall = sum(1 for s in scenarios if not s["is_baseline"] and s["wall_r_value"] and not s["roof_r_value"] and not s["window_u_factor"] and not s["door_option"])
    individual_roof = sum(1 for s in scenarios if not s["is_baseline"] and s["roof_r_value"] and not s["wall_r_value"] and not s["window_u_factor"] and not s["door_option"])
    individual_window = sum(1 for s in scenarios if not s["is_baseline"] and s["window_u_factor"] and not s["wall_r_value"] and not s["roof_r_value"] and not s["door_option"])
    individual_door = sum(1 for s in scenarios if not s["is_baseline"] and s["door_option"] and not s["wall_r_value"] and not s["roof_r_value"] and not s["window_u_factor"])
    all_measures = sum(1 for s in scenarios if not s["is_baseline"] and sum([bool(s["wall_r_value"]), bool(s["roof_r_value"]), bool(s["window_u_factor"]), bool(s["door_option"])]) > 1)

    print(f"\n📦 Total scenarios: {total_sims}")
    print(f"   - Cities: {len(CITIES)}")
    print(f"   - Building Types: {len(BUILDING_TYPES)}")
    print(f"\n   Breakdown:")
    print(f"   - Baseline: {baseline_count}")
    print(f"   - Wall only: {individual_wall}")
    print(f"   - Roof only: {individual_roof}")
    print(f"   - Window only: {individual_window}")
    print(f"   - Door only: {individual_door}")
    print(f"   - Combined (≥2 measures): {all_measures}")
    print("=" * 70)

    sim_count = 0
    start_time = time.time()
    successful_scenarios = []
    failed_scenarios = []

    for scenario in scenarios:
        sim_count += 1
        city = scenario["city"]
        building_type = scenario["building_type"]
        climate_zone = city_climate_zones.get(city, "ASHRAE 169-2013-5A")

        scenario_name = generate_scenario_name(scenario)
        print(f"\n[{sim_count}/{total_sims}] {scenario_name}")

        sim_start = time.time()

        result = create_simulation(
            city=city,
            base_run_dir=base_run_dir,
            measure_dir_path=measure_dir_path,
            base_weather_path=base_weather_path,
            scenario_dict=scenario,
            overwrite_existing=OVERWRITE_EXISTING,
            building_type=building_type,
            template=TEMPLATE,
            climate_zone=climate_zone,
            openstudio_path=OPENSTUDIO_PATH,
        )

        if result:
            successful_scenarios.append(result)
        else:
            failed_scenarios.append(scenario_name)

        sim_elapsed = time.time() - sim_start
        print(f"   ⏱️  Time: {sim_elapsed/60:.1f} min")

    total_elapsed = time.time() - start_time

    print("\n" + "=" * 70)
    print("SIMULATION SUMMARY")
    print("=" * 70)
    print(f"⏱️  Total time: {total_elapsed/60:.1f} min ({total_elapsed/3600:.2f} hours)")
    print(f"✅ Successful: {len(successful_scenarios)}/{total_sims}")
    print(f"❌ Failed: {len(failed_scenarios)}/{total_sims}")

    if failed_scenarios:
        print("\nFailed scenarios:")
        for failed in failed_scenarios:
            print(f"  - {failed}")

    # Collect results
    print("\n" + "=" * 70)
    print("COLLECTING RESULTS FROM OSM FILES")
    print("=" * 70)
    generate_parametric_recap(f"./simulations/{RUN_NAME}")
    print("\n✅ Parametric study complete!")



PARAMETRIC STUDY: BUILDING ENERGY EFFICIENCY MEASURES
Using OpenStudio: C:/Program Files/openstudio-3.11.0/bin/openstudio.exe
Run Name: run_test_001
Output Directory: c:\All repos\openstudio-ee-gem\lib\parametric_run\simulations\run_test_001

📋 Generating scenarios...

📦 Total scenarios: 4
   - Cities: 1
   - Building Types: 1

   Breakdown:
   - Baseline: 1
   - Wall only: 0
   - Roof only: 0
   - Window only: 0
   - Door only: 0
   - Combined (≥2 measures): 3

[1/4] baseline_SmallOffice_Amarillo
  Applying reporting measure...
✅ Completed: baseline_SmallOffice_Amarillo
   ⏱️  Time: 0.4 min

[2/4] all_wall_r30_door_wooden_d_SmallOffice_Amarillo
  Applying wall insulation (R=30)...
  Applying door enhancement (door=wooden door)...
{'Perimeter_ZN_1_wall_south_door': {'Door type': 'GlassDoor',
                                    'dimension': {'area_m2': 3.90522,
                                                  'length_m': 2.134,
                                                  'perime